# NeedLE AI: Agentic RAG System for CMS Medical Coverage Decision Support

**GitHub-ready version.** Outputs were cleared and API keys were removed.  
Before running, add required CMS source files to your local/Colab environment and set `OPENAI_API_KEY` securely using environment variables or Colab Secrets.


# PROJECT: NeedLE AI
This notebook implements the NeedLe (Need for Lab Evidence) Medical Necessity Checker by combining to evaluate laboratory test coverage and explain relevant policy evidence:
- CMS NCD policy extraction
- CPT–ICD coverage logic
- LangChain/LangGraph agent workflows



# IMPORTS & SETUP

In [ ]:
# Install the required Python packages/libraries
!pip install -q -U pypdf langchain langchain-openai langchain-community langgraph pyarrow

In [ ]:
# Import libraries
import os
import re
import json
import numpy as np
import pandas as pd
from pypdf import PdfReader

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

from typing import TypedDict, List, Dict, Any
from IPython.display import Image, display

In [ ]:
# --------------------------------------------------
# Setup and Paths
# --------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")


base_path = "/content/drive/MyDrive/Colab files/"
save_path = "/content/drive/MyDrive/NeedLe_outputs"


# I. DATA PREPARATION AND COVERAGE LOGIC


## Define file paths

In [ ]:
# Policy manual with 23 Lab policies
pdf_path = os.path.join(base_path, "2026100-ICD-10-NCD-Manual-20251217-508.pdf")

# CPT to ICD link-coverage file
excel_path = os.path.join(base_path, "2026100-Initial-ICD10-NCD-Spreadsheet-20250721.xlsx")

#ICD Descriptions file
icd_path = os.path.join(base_path, "icd10cm_order_2026.txt")

# CPT descriptons file
cpt_path = os.path.join(base_path, "CPT Descriptions.xlsx")


## File 1: Load, Process, and Parse PDF Lab Policy Manual


- Extract all bookmarks to identify NCD policy sections and subsections (e.g., Description, Indications, Limitations) with Page numbers
- Remove repeated headers/footer content (clean boilerplate) to remove noise
- Filter Extracted Policy Records
- Build a clean NCD Index for use in the RAG retrieval system.



In [ ]:
# ----------------------------------------------------------
# 1. Parse CMS NCD PDF POLICY MANUAL
# ----------------------------------------------------------
reader = PdfReader(pdf_path)
num_pages_total = len(reader.pages)
print("Total PDF pages:", num_pages_total)


In [ ]:
# ==========================================================
# POLICY SECTIONS & SUB-SECTIONS EXTRACTION WITH CLEANING
# NCD identification, filtering, boilerplate cleaning
# ==========================================================

# Flatten outline (bookmarks) into reading-order items

def flatten_outline(outline, level=0):
    """
    Returns a flat list of dicts: {title, page, level}
    page is 0-based.
    """
    items = []
    for obj in outline:
        if isinstance(obj, list):
            items.extend(flatten_outline(obj, level=level+1))
        else:
            try:
                title = (obj.title or "").strip()
                page_idx = reader.get_destination_page_number(obj)  # 0-based
                items.append({"title": title, "page": page_idx, "level": level})
            except Exception:
                continue
    return items

outline_items = flatten_outline(reader.outline)
outline_items = sorted(outline_items, key=lambda x: (x["page"], x["level"]))

print("Total bookmark items:", len(outline_items))
print("Sample bookmarks:")
for it in outline_items[:20]:
    print(f"{it['title']} (page {it['page']})")


# Identify NCD headings + normalize section names

NCD_RE = re.compile(r"^\s*(\d+(?:\.\d+)*)\s*-\s*(.+?)\s*$")

def parse_ncd_heading(title: str):
    m = NCD_RE.match(title)
    if not m:
        return None
    return {"ncd_number": m.group(1), "ncd_title": m.group(2).strip()}

def normalize_section(title: str) -> str:
    return re.sub(r"\s+", " ", (title or "")).strip()

# Build NCD index

ncd_index = []
seen = set()

for item in outline_items:
    parsed = parse_ncd_heading(item["title"])
    if parsed and parsed["ncd_number"] not in seen:
        seen.add(parsed["ncd_number"])
        ncd_index.append({
            "ncd_number": parsed["ncd_number"],
            "ncd_title": parsed["ncd_title"],
            "start_page": item["page"]
        })

# Extract text by page range

def extract_text_pages(start_page: int, end_page_exclusive: int) -> str:
    parts = []
    for p in range(start_page, end_page_exclusive):
        parts.append(reader.pages[p].extract_text() or "")
    return "\n".join(parts).strip()


# Compute the first NCD start page from ncd_index

first_ncd_page = min(n["start_page"] for n in ncd_index) if ncd_index else None
print("First NCD starts at page:", first_ncd_page)


print("Total NCDs:", len(ncd_index))
print("\nNCD List:\n")

for n in ncd_index:
    print(f"{n['ncd_number']} - {n['ncd_title']} (Start Page: {n['start_page']})")

# Section detection logic

SECTION_PATTERNS = {
    "Other Names/Abbreviations": r"(?im)^Other Names/?Abbreviations",
    "Description": r"(?im)^Description",
    "HCPCS Codes": r"(?im)^HCPCS Codes",
    "ICD-10-CM Codes Covered by Medicare Program": r"(?im)^ICD-10-CM Codes Covered",
    "Indications": r"(?im)^Indications",
    "Limitations": r"(?im)^Limitations",
    "ICD-10-CM Codes That Do Not Support Medical Necessity": r"(?im)^ICD-10-CM Codes That Do Not Support",
    "Documentation Requirements": r"(?im)^Documentation Requirements",
    "Other Comments": r"(?im)^Other Comments",
    "Sources of Information": r"(?im)^Sources of Information",
}

def split_ncd_sections(text):
    sections = {}

    for name, pattern in SECTION_PATTERNS.items():
        match = re.search(pattern, text)
        if match:
            sections[name] = match.start()

    ordered = sorted(sections.items(), key=lambda x: x[1])

    results = {}
    for i, (name, start) in enumerate(ordered):
        end = ordered[i+1][1] if i+1 < len(ordered) else len(text)
        results[name] = text[start:end].strip()

    return results



# Find which NCD a bookmark belongs to (simple backward scan)

def find_current_ncd(outline_items, idx, first_ncd_page=None):
    """
    Find nearest preceding NCD heading for a bookmark item.
    Guard: do not assign NCD info to bookmarks before the first NCD page.
    """
    cur = outline_items[idx]
    cur_title = cur["title"]
    cur_page = cur["page"]

    # Guard: anything before first NCD is global (Introduction, etc.)
    if first_ncd_page is not None and cur_page < first_ncd_page:
        return None

    # If current is an NCD heading itself
    parsed = parse_ncd_heading(cur_title)
    if parsed:
        return parsed

    # Otherwise scan backwards for latest NCD heading
    for j in range(idx - 1, -1, -1):
        maybe = parse_ncd_heading(outline_items[j]["title"])
        if maybe and outline_items[j]["page"] <= cur_page:
            return maybe

    return None


# Build subsection records (bookmark → next bookmark page)
def build_section_records(outline_items, first_ncd_page=None):

    records = []
    seen_sections = set()
    num_pages_total = len(reader.pages)

    for i, item in enumerate(outline_items):

        title = normalize_section(item["title"])
        start = item["page"]

        end = outline_items[i+1]["page"] if i+1 < len(outline_items) else num_pages_total

        # Allow same-page bookmarks
        if end < start:
            continue

        # Ensure at least one page extracted
        extract_end = max(end, start + 1)

        raw_text = extract_text_pages(start, extract_end)

        if not raw_text:
            continue

        # Skip NCD heading containers
        if parse_ncd_heading(title):
            continue

        # Identify parent NCD
        ncd_info = find_current_ncd(outline_items, i, first_ncd_page=first_ncd_page)

        if ncd_info:

            # Split text into actual sections
            sections = split_ncd_sections(raw_text)

            for sec_name, sec_text in sections.items():

                key = (ncd_info["ncd_number"], sec_name)

                if key in seen_sections:
                    continue

                seen_sections.add(key)

                records.append({
                    "doc_type": "ncd",
                    "ncd_number": ncd_info["ncd_number"],
                    "ncd_title": ncd_info["ncd_title"],
                    "section": sec_name,
                    "start_page": start,
                    "end_page": end - 1,
                    "text": sec_text
                })

        else:
            # Global section
            records.append({
                "doc_type": "global",
                "section": title,
                "start_page": start,
                "end_page": end - 1,
                "text": raw_text,
            })

    return records



records = build_section_records(outline_items, first_ncd_page=first_ncd_page)

print("\nTotal subsection records:", len(records))
print("Global records:", sum(r["doc_type"] == "global" for r in records))
print("NCD records:", sum(r["doc_type"] == "ncd" for r in records))

# Show a few important globals if present
for key in [
    "Non-covered ICD-10-CM Codes for All Lab NCDs",
    "Reasons for Denial for All Lab NCDs",
    "Coding Guidelines for All Lab NCDs",
    "Additional Coding Guideline(s)",
]:
    hits = [r for r in records if r["doc_type"] == "global" and key.lower() in (r["section"] or "").lower()]
    if hits:
        h = hits[0]
        print("\nFOUND:", key, "->", h["section"], "pages", h["start_page"], "-", h["end_page"])


#5. Filter relevant policy sections

VERBIAGE_NCD_SECTIONS = {
    "Other Names/Abbreviations",
    "Description",
    "Indications",
    "Limitations",
    "Documentation Requirements",
    "Other Comments",
    "Sources of Information",
}


filtered_records = [
    r for r in records
    if r["doc_type"] == "global"
    or (r["doc_type"] == "ncd" and r["section"] in VERBIAGE_NCD_SECTIONS)
]

print("Filtered records (Global + NCD verbiage):", len(filtered_records))


#6 Clean header/footer boilerplate (Removes page headers/footers, Important for LLM context quality)

import re

def clean_boilerplate(text: str) -> str:
    if not text:
        return text

    patterns = [
        r"Medicare National Coverage Determinations.*?\n",
        r"Coding Policy Manual.*?\n",
        r"Fu Associates, Ltd\..*?\n",
        r"\*January 2026 Changes.*?\n",
        r"ICD-10-CM Version.*?\n",
    ]

    cleaned = text
    for pat in patterns:
        cleaned = re.sub(pat, "", cleaned, flags=re.IGNORECASE)

    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip()
    return cleaned

# Apply cleaning
for r in filtered_records:
    r["text"] = clean_boilerplate(r["text"])

print("Boilerplate cleaned.")



Output Explanation:
- Successful extraction of all sections, subsections, their page numbers and building of NCD index.
- Total bookmark items: 173
- Total NCDs extracted: 23
- Total subsection records: 197
- Global records: 21

In [ ]:
# -----------------------------------------
# Output: Extracted Sections Per NCD
# -----------------------------------------

from collections import defaultdict

ncd_sections = defaultdict(set)

for r in filtered_records:
    if r["doc_type"] == "ncd":
        ncd_sections[r["ncd_number"]].add(r["section"])

print("Extracted Sections by NCD:\n")

for ncd in sorted(ncd_sections.keys()):
    sections = sorted(list(ncd_sections[ncd]))
    print(f"NCD {ncd}: {sections}")

## File 2-Load and Process Coverage Dataset (CPT+ICD Excel Coverage table) for logic

- Loads the CMS CPT–ICD coverage table from Excel
- Standardizes and cleans the column names, dates, and diagnosis codes
- Define Coverage Criteria for NCD Policies that determines whether an ICD-10 diagnosis code is covered under a specific NCD policy, optionally validating the code against effective and termination dates to simulate real CMS claim coverage logic.
- Prepare the dataset used by the rule-based coverage decision logic in the system.



In [ ]:
import pandas as pd
import numpy as np

# Load coverage table
df = pd.read_excel(excel_path, sheet_name=0, dtype=str)

# Normalize column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
)

# Rename to internal schema
df = df.rename(columns={
    "cpt_hcpcs_codes": "cpt_code",
    "cpt_hcpcs_eff_date": "cpt_eff_date",
    "cpt_hcpcs_term_date": "cpt_term_date",
    "icd-10_code": "icd_code",
    "icd-10_eff__date": "icd_eff_date",
    "icd-10_term_date": "icd_term_date",
})

# Convert date columns
date_cols = ["cpt_eff_date","cpt_term_date","icd_eff_date","icd_term_date"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Normalize key fields
df["ncd_number"] = df["ncd_number"].astype(str).str.strip()
df["cpt_code"] = df["cpt_code"].astype(str).str.strip()
df["icd_code_norm"] = df["icd_code"].str.upper().str.strip()
df["icd_code_nodot"] = df["icd_code_norm"].str.replace(".", "", regex=False)

# Resolution code
df["resolution_code"] = pd.to_numeric(df["resolution_code"], errors="coerce")

print("Coverage table shape:", df.shape)
print("Columns:", df.columns.tolist())

#-----------------------------------------------
#Define Coverage Criteria for NCD Policies
#-----------------------------------------------

RESOLUTION_MEANING = {
    1: "COVERED",
    2: "DENIED",
    3: "DOES NOT SUPPORT MEDICAL NECESSITY"
}

def check_coverage(df, ncd_number, icd_code, date_of_service=None):

    ncd_number = str(ncd_number).strip()
    icd_code = str(icd_code).strip().upper()
    icd_nodot = icd_code.replace(".", "")

    x = df[df["ncd_number"] == ncd_number]

    if x.empty:
        return {"found": False, "reason": f"NCD {ncd_number} not found"}

    # ICD match
    x = x[(x["icd_code_norm"] == icd_code) | (x["icd_code_nodot"] == icd_nodot)]

    if x.empty:
        return {"found": False, "reason": f"ICD {icd_code} not found under NCD {ncd_number}"}

    # Date filtering
    if date_of_service:
        dos = pd.to_datetime(date_of_service)

        x = x[
            (x["icd_eff_date"].isna() | (x["icd_eff_date"] <= dos)) &
            (x["icd_term_date"].isna() | (dos <= x["icd_term_date"]))
        ]

        if x.empty:
            return {"found": False, "reason": f"ICD {icd_code} not active on {date_of_service}"}

    row = x.sort_values("icd_eff_date", ascending=False).iloc[0]

    resolution = int(row["resolution_code"])

    return {
        "found": True,
        "ncd_number": ncd_number,
        "icd_code": icd_code,
        "resolution_code": resolution,
        "meaning": RESOLUTION_MEANING.get(resolution, "UNKNOWN"),
        "icd_eff_date": row["icd_eff_date"],
        "icd_term_date": row["icd_term_date"],
    }

required_cols = {"ncd_number", "icd_code", "resolution_code"}
missing = required_cols - set(df.columns)
print("Coverage table schema OK." if not missing else f"WARNING: Missing columns: {missing}")

## File 3- Load and Process ICD Reference Dataset
Dataset to validate diagnosis codes and retrieve official ICD descriptions.
- Loads the CMS ICD-10-CM order file
- parses to extract diagnosis codes and descriptions
- filters valid billable codes
- adds the standard dotted ICD format
- implements lookup utilities that allow ICD codes to be retrieved either directly by code or through description-based search for use in coverage evaluation and reporting.


In [ ]:
# -------------------------------------------------------
# Load File 3: ICD-10 Reference File
# -------------------------------------------------------

def parse_icd10cm_order_line(line: str):

    if not line or len(line) < 78:
        return None

    code_nodot = line[6:13].strip()
    valid_flag = line[14:15].strip()
    short_desc = line[16:76].strip()
    long_desc = line[77:].strip()

    if not code_nodot:
        return None

    return {
        "icd_code_nodot": code_nodot,
        "valid_flag": valid_flag,
        "short_desc": short_desc,
        "long_desc": long_desc
    }

rows = []

with open(icd_path, "r", encoding="latin-1") as f:
    for line in f:
        rec = parse_icd10cm_order_line(line.rstrip("\n"))
        if rec:
            rows.append(rec)

icd_df = pd.DataFrame(rows)

icd_valid = icd_df[icd_df["valid_flag"] == "1"].copy()

print("Total ICD rows:", len(icd_df))
print("Valid ICD rows:", len(icd_valid))

#Add dotted ICD formatting
def add_icd_dot(code_nodot):
    c = str(code_nodot).strip().upper()
    if len(c) <= 3:
        return c
    return c[:3] + "." + c[3:]

icd_valid["icd_code"] = icd_valid["icd_code_nodot"].apply(add_icd_dot)

#ICD lookup functions
def icd_lookup(icd_code):
    nodot = icd_code.replace(".", "")
    hit = icd_valid.query("icd_code_nodot == @nodot")

    if hit.empty:
        return None

    return hit.iloc[0].to_dict()

## File 4- Load CPT Reference Data
- Provides CPT descriptions
- Maps CPT codes to their associated NCD policies


In [ ]:
# -------------------------------------------------------
# Load File 4: CPT Reference File
# -------------------------------------------------------

cpt_df = pd.read_excel(cpt_path)

# Standardize columns
cpt_df.columns = (
    cpt_df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace("#", "number", regex=False)
    .str.replace(" ", "_", regex=False)
)

# Rename CMS field
cpt_df = cpt_df.rename(columns={
    "ncdnumber": "ncd_number"
})

# Clean fields
cpt_df["ncd_number"] = cpt_df["ncd_number"].astype(str).str.strip()
cpt_df["cpt_code"] = cpt_df["cpt_code"].astype(str).str.strip()
cpt_df["cpt_description"] = cpt_df["cpt_description"].astype(str).str.strip()

print("Loaded CPT file:", cpt_df.shape)
display(cpt_df.head())


# Build CPT → description lookup
cpt_desc_lookup = dict(
    zip(cpt_df["cpt_code"], cpt_df["cpt_description"])
)

# Build CPT → NCD mapping
cpt_to_ncd = (
    cpt_df.groupby("cpt_code")["ncd_number"]
    .apply(list)
    .to_dict()
)

print("Total CPT descriptions:", len(cpt_desc_lookup))
print("Total CPT → NCD mappings:", len(cpt_to_ncd))

## Policy Retrieval Functions
- Retrieves the most relevant policy sections such as indications and limitations using score matching in RAG system.


In [ ]:
# ==========================================================
# Retrieval layer
# ==========================================================
import re
# -----------------------------------
# Build NCD Title Lookup
# -----------------------------------

ncd_title_lookup = {}

for r in filtered_records:
    if r.get("doc_type") == "ncd":
        ncd_num = r.get("ncd_number")
        ncd_title = r.get("ncd_title")

        if ncd_num and ncd_title:
            ncd_title_lookup[ncd_num] = ncd_title

print("Total NCD titles loaded:", len(ncd_title_lookup))

def find_best_ncd(query: str, records):
    """
    Returns the best matching NCD number (string) using NCD title + text evidence.
    """
    q = query.lower().strip()
    q_terms = [t for t in re.split(r"\W+", q) if t]

    ncd_scores = {}
    ncd_titles = {}

    for r in records:
        if r.get("doc_type") != "ncd":
            continue

        ncd_no = r.get("ncd_number")
        ncd_title = (r.get("ncd_title") or "").lower()
        text = (r.get("text") or "").lower()

        # Save original title for later display
        if ncd_no and ncd_no not in ncd_titles:
            ncd_titles[ncd_no] = r.get("ncd_title")

        score = 0
        for t in q_terms:
            if t in ncd_title:
                score += 6          # strongest signal
            if t in text:
                score += 1

        if score > 0 and ncd_no:
            ncd_scores[ncd_no] = max(ncd_scores.get(ncd_no, 0), score)

    if not ncd_scores:
        return None

    best_ncd = max(ncd_scores.items(), key=lambda x: x[1])[0]
    return best_ncd


PREFERRED_ORDER = [
    "Description",
    "Indications",
    "Limitations",
    "Documentation Requirements",
    "Other Comments",
    "Other Names/Abbreviations",
]

def get_ncd_bundle(ncd_number: str, records):
    """
    Returns curated sections for one NCD in a fixed order.
    """
    out = []
    for sec in PREFERRED_ORDER:
        hit = next(
            (r for r in records
             if r.get("doc_type") == "ncd"
             and r.get("ncd_number") == ncd_number
             and r.get("section") == sec),
            None
        )
        if hit:
            out.append(hit)
    return out


def get_policy_for_test(query: str, records):
    """
    Main API: given a test name/description, return best NCD + curated sections.
    """
    best_ncd = find_best_ncd(query, records)
    if not best_ncd:
        return {"best_ncd": None, "sections": []}

    bundle = get_ncd_bundle(best_ncd, records)
    return {"best_ncd": best_ncd, "sections": bundle}

print("Policy retrieval layer ready.")
print("Filtered policy records:", len(filtered_records))

## Integrating Coverage Rules with Policy Evidence
This step combines coverage rules with policy retrieval.

- Checks whether a CPT and ICD combination is covered based on CMS rules
- Then retrieves supporting policy text to justify the coverage decision

The hybrid rule-based and retrieval-augmented component of the system.

In [ ]:
# -------------------------------------------------------
# Link Coverage Rules with Policy Evidence
# -------------------------------------------------------
#Add helpers to pull NCD verbiage sections + global denial reasons

def get_global_section(records, contains_text: str):
    contains_text = contains_text.lower()
    hits = [r for r in records if r.get("doc_type")=="global" and contains_text in (r.get("section") or "").lower()]
    return hits[0] if hits else None

def get_ncd_section(records, ncd_number: str, section: str):
    for r in records:
        if r.get("doc_type")=="ncd" and r.get("ncd_number")==ncd_number and r.get("section")==section:
            return r
    return None

# Combined coverage + policy retrieval function
def evaluate_test_coverage(test_name: str, icd_code: str, date_of_service: str,
                           verbiage_records, coverage_df):
    # 1) Find best NCD from test name
    policy_pick = get_policy_for_test(test_name, verbiage_records)
    best_ncd = policy_pick["best_ncd"]

    if not best_ncd:
        return {
            "test_name": test_name,
            "icd_code": icd_code,
            "date_of_service": date_of_service,
            "found_ncd": False,
            "message": "Could not identify an NCD for this test name."
        }

    # 2) Check deterministic coverage from Excel
    cov = check_coverage(coverage_df, best_ncd, icd_code, date_of_service)

    # 3) Pull the most useful verbiage
    indications = get_ncd_section(verbiage_records, best_ncd, "Indications")
    limitations = get_ncd_section(verbiage_records, best_ncd, "Limitations")
    description = get_ncd_section(verbiage_records, best_ncd, "Description")

    denial_global = None
    if cov.get("found") and cov.get("resolution_code") in {2, 3}:
        denial_global = get_global_section(verbiage_records, "Reasons for Denial")

    return {
        "test_name": test_name,
        "icd_code": icd_code,
        "date_of_service": date_of_service,
        "found_ncd": True,
        "ncd_number": best_ncd,
        "coverage": cov,
        "verbiage": {
            "description": description["text"] if description else None,
            "indications": indications["text"] if indications else None,
            "limitations": limitations["text"] if limitations else None,
            "global_reasons_for_denial": denial_global["text"] if denial_global else None,
        }
    }


# -------------------------------------------------------
# CPT + ICD Coverage Evaluation Function
# -------------------------------------------------------

def check_coverage_by_cpt_icd(
    cpt_code: str,
    icd_code: str,
    date_of_service: str,
    df_cov,
    cpt_to_ncd_map
):
    """
    Evaluate CPT–ICD coverage by checking all NCD policies linked to the CPT.
    """

    cpt_code = str(cpt_code).strip()
    candidates = cpt_to_ncd_map.get(cpt_code, [])

    if not candidates:
        return {
            "found": False,
            "reason": f"No NCD found for CPT {cpt_code}."
        }

    results = []

    for ncd in candidates:
        res = check_coverage(df_cov, ncd, icd_code, date_of_service)

        # attach NCD number safely
        res = {**res, "ncd_number": ncd}

        results.append(res)

    # Rank results:
    # 1) found coverage
    # 2) covered codes
    results_sorted = sorted(
        results,
        key=lambda r: (
            0 if r.get("found") else 1,
            0 if r.get("resolution_code") == 1 else 1
        )
    )

    return {
        "found": True,
        "cpt_code": cpt_code,
        "icd_code": icd_code,
        "date_of_service": date_of_service,
        "results": results_sorted
    }


# II. BUILD LANGCHAIN AGENT FOR COVERAGE DECISION SUPPORT
- This step converts the coverage functions into LangChain tools.
- The agent interprets user queries and calls the appropriate tools to retrieve coverage decisions and supporting policy explanations.

## Imports and model setup

In [ ]:
# Initialize OpenAI chat model
# SECURITY NOTE:
# Do NOT hard-code API keys in notebooks. Set OPENAI_API_KEY as an environment variable
# or use Colab Secrets before running this cell.

import os
from langchain_openai import ChatOpenAI

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is not set. Add it as an environment variable or Colab Secret before running."
    )

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.1, max_tokens=512)


## Tool Definitions

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
import json

# -------------------------------
# Tool Definitions
# -------------------------------

@tool
def coverage_by_cpt_icd(cpt_code: str, icd_code: str, date_of_service: str) -> str:
    """Check coverage using CPT and ICD."""
    result = check_coverage_by_cpt_icd(
        cpt_code,
        icd_code,
        date_of_service,
        df,
        cpt_to_ncd
    )
    return json.dumps(result, indent=2, default=str)


@tool
def evaluate_lab_coverage(test_name: str, icd_code: str, date_of_service: str) -> str:
    """End-to-end coverage evaluation using test name."""
    result = evaluate_test_coverage(
        test_name=test_name,
        icd_code=icd_code,
        date_of_service=date_of_service,
        verbiage_records=filtered_records,
        coverage_df=df
    )
    return json.dumps(result, indent=2, default=str)


@tool
def lookup_icd_description(icd_code: str) -> str:
    """Retrieve ICD description."""
    result = icd_lookup(icd_code, icd_valid)

    if result is None:
        return json.dumps({"found": False})

    return json.dumps(result, indent=2)


tools = [
    coverage_by_cpt_icd,
    evaluate_lab_coverage,
    lookup_icd_description
]

print("Tools loaded:", [t.name for t in tools])



## System Prompt and Agent Initialization

In [ ]:

# -------------------------------
# System Prompt
# -------------------------------

system_prompt = """
You are a Lab NCD Coverage Assistant.

Your role is to help determine whether a laboratory test may be covered under a Medicare
National Coverage Determination (NCD) policy. You must rely only on the structured
coverage logic derived from the CMS coverage spreadsheet and the policy verbiage
extracted from the NCD Lab Policy Manual.

Guidelines:
1. Use professional and neutral language in all responses.
2. Use available tools whenever the question involves CPT codes, ICD-10 codes,
   NCD policies, or coverage rules.
3. Prefer deterministic coverage results from the coverage spreadsheet when available.
4. Use NCD policy verbiage only to support or explain the coverage decision.
5. Do not invent CPT, ICD, or NCD codes.
6. If required information (such as CPT code, ICD code, or date of service) is missing,
   clearly explain what information is needed.
7. Base all answers strictly on the provided knowledge sources and tools.
8. Do not provide personal opinions, clinical advice, or medical recommendations.

When presenting a final answer, include:
• Identified NCD policy (if found)
• Coverage determination (e.g., Covered, Denied, Does Not Support Medical Necessity)
• Brief explanation of the coverage logic
• Relevant policy language such as Indications or Limitations when available

If the system cannot determine coverage from the available data, politely explain
that the information could not be determined and suggest providing additional
details such as CPT code, ICD-10 diagnosis, or date of service.
"""


# -------------------------------
# Create Agent
# -------------------------------

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)


# -------------------------------
# Helper Function to Query Agent
# -------------------------------

def ask_agent(question):
    response = agent.invoke(
        {"messages":[{"role":"user","content":question}]}
    )
    return response["messages"][-1].content


print("LangChain tools initialized successfully.")
print("Ready for LangGraph workflow.")

# III. BUILD LANGRAPH
*   State
*   Helper
*   Nodes
*   Build Graph




## Build LangGraph Workflow
- query parsing
- evidence retrieval
- coverage decision
- confidence scoring
- final response generation

In [ ]:
import re
import json
import pandas as pd
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


# ---------------------------------------------------------
# STATE
# ---------------------------------------------------------

class NeedLeState(TypedDict):
    question: str
    intent: str
    cpt_code: str
    icd_code: str
    date_of_service: str
    test_name: str
    evidence: List[Dict[str, Any]]
    coverage_result: Dict[str, Any]
    confidence: Dict[str, Any]
    final_answer: str


# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------

KNOWN_TESTS = [
    "urine culture",
    "blood culture",
    "lipid panel",
    "glucose",
    "hemoglobin a1c",
    "a1c",
    "thyroid stimulating hormone",
    "tsh",
    "vitamin d test",
    "hiv-1 testing",
    "serum iron studies",
    "iron studies",
    "iron test"
]


def extract_fields(question):

    text = question.lower()

    cpt = re.search(r"\b(\d{5})\b", text)
    icd = re.search(r"\b([a-tv-z][0-9]{2}(?:\.[0-9a-z]{1,4})?)\b", text)
    date = re.search(r"\b(20\d{2}-\d{2}-\d{2})\b", text)

    test = ""

    for t in KNOWN_TESTS:
        if t in text:
            test = t
            break

    return {
        "cpt_code": cpt.group(1) if cpt else "",
        "icd_code": icd.group(1).upper() if icd else "",
        "date_of_service": date.group(1) if date else "",
        "test_name": test
    }


def classify_intent(text: str):

    t = text.lower()

    if any(k in t for k in ["covered", "coverage", "medical necessity"]):
        return "coverage"

    if any(k in t for k in ["policy", "indication", "limitation", "ncd"]):
        return "policy"

    return "general"


def retrieve_evidence(test_name: str):

    policy = get_policy_for_test(test_name, filtered_records)

    evidence = []

    if not policy or not policy.get("best_ncd"):
        return evidence

    ncd = policy["best_ncd"]

    for sec in ["Description", "Indications", "Limitations"]:
        rec = get_ncd_section(filtered_records, ncd, sec)

        if rec and rec.get("text"):
            evidence.append({
                "ncd": ncd,
                "section": sec,
                "text": rec["text"][:500]
            })

    return evidence


def compute_confidence(result, evidence, fields):

    score = 0

    if fields["cpt_code"]:
        score += 20
    if fields["icd_code"]:
        score += 20
    if fields["date_of_service"]:
        score += 20
    if evidence:
        score += 20
    if result.get("found"):
        score += 20

    if score >= 80:
        label = "high"
    elif score >= 50:
        label = "medium"
    else:
        label = "low"

    return {"score": score, "label": label}


# ---------------------------------------------------------
# NODES
# ---------------------------------------------------------

def parse_node(state: NeedLeState):

    fields = extract_fields(state["question"])
    intent = classify_intent(state["question"])

    return {
        "intent": intent,
        "cpt_code": fields["cpt_code"],
        "icd_code": fields["icd_code"],
        "date_of_service": fields["date_of_service"],
        "test_name": fields["test_name"],
    }


def evidence_node(state: NeedLeState):

    evidence = retrieve_evidence(state.get("test_name", ""))

    return {"evidence": evidence}


def decision_node(state: NeedLeState):

    cpt = (state.get("cpt_code") or "").strip()
    icd = (state.get("icd_code") or "").strip()
    dos = (state.get("date_of_service") or "").strip()
    test = (state.get("test_name") or "").strip()
    intent = (state.get("intent") or "").strip().lower()

    try:
        # 1. Policy-only question
        if intent == "policy":
            if not test:
                result = {
                    "found": False,
                    "error": "Policy question detected, but no recognizable test name was found."
                }
            else:
                policy = get_policy_for_test(test, filtered_records)

                if not policy or not policy.get("best_ncd"):
                    result = {
                        "found": False,
                        "error": f"No matching NCD policy found for test '{test}'."
                    }
                else:
                    ncd = policy["best_ncd"]
                    result = {
                        "found": True,
                        "ncd_number": ncd,
                        "description": (get_ncd_section(filtered_records, ncd, "Description") or {}).get("text", "")[:1500],
                        "indications": (get_ncd_section(filtered_records, ncd, "Indications") or {}).get("text", "")[:1500],
                        "limitations": (get_ncd_section(filtered_records, ncd, "Limitations") or {}).get("text", "")[:1500],
                    }

        # 2. CPT + ICD + date path
        elif cpt and icd and dos:
            result = check_coverage_by_cpt_icd(
                cpt,
                icd,
                dos,
                df,
                cpt_to_ncd
            )

        # 3. Test name + ICD + date path
        elif test and icd and dos:
            result = evaluate_test_coverage(
                test,
                icd,
                dos,
                filtered_records,
                df
            )

        # 4. General question
        elif intent == "general":
            result = {
                "found": True,
                "message": "NeedLe can help evaluate lab test coverage, retrieve NCD policy sections, and explain coverage decisions using CPT, ICD-10, and date of service."
            }

        # 5. Missing inputs
        else:
            result = {
                "found": False,
                "error": "Insufficient inputs. Provide CPT code or test name, ICD-10 code, and date of service for a coverage decision."
            }

    except Exception as e:
        result = {"found": False, "error": str(e)}

    return {"coverage_result": result}



def confidence_node(state: NeedLeState):

    fields = {
        "cpt_code": state.get("cpt_code"),
        "icd_code": state.get("icd_code"),
        "date_of_service": state.get("date_of_service"),
        "test_name": state.get("test_name"),
    }

    conf = compute_confidence(
        state.get("coverage_result", {}),
        state.get("evidence", []),
        fields
    )

    return {"confidence": conf}


def final_node(state: NeedLeState):

    greeting = "Hi! Welcome to NeedLe — the Need for Lab Evidence coverage assistant.\n\n"

    cov = state.get("coverage_result", {})
    nested_cov = cov.get("coverage", {})
    results_list = cov.get("results", [])
    first_result = results_list[0] if results_list else {}

    ncd_number = (
        cov.get("ncd_number")
        or nested_cov.get("ncd_number")
        or first_result.get("ncd_number")
        or ""
    )

    ncd_line = ""
    if ncd_number:
        ncd_title = ncd_title_lookup.get(ncd_number, "Unknown Title")
        ncd_line = f"Identified NCD: {ncd_number} – {ncd_title}\n\n"

    text = (
        greeting
        + f"Question: {state['question']}\n\n"
        + ncd_line
        + f"Coverage Result:\n{json.dumps(cov, indent=2, default=str)}\n\n"
        + f"Confidence: {state.get('confidence', {}).get('label')} "
        + f"({state.get('confidence', {}).get('score')}/100)"
    )

    return {"final_answer": text}



# ---------------------------------------------------------
# BUILD GRAPH
# ---------------------------------------------------------

workflow = StateGraph(NeedLeState)

workflow.add_node("parse", parse_node)
workflow.add_node("evidence", evidence_node)
workflow.add_node("decision", decision_node)
workflow.add_node("confidence", confidence_node)
workflow.add_node("final", final_node)

workflow.add_edge(START, "parse")
workflow.add_edge("parse", "evidence")
workflow.add_edge("evidence", "decision")
workflow.add_edge("decision", "confidence")
workflow.add_edge("confidence", "final")
workflow.add_edge("final", END)

memory = InMemorySaver()

graph = workflow.compile(checkpointer=memory)

## Visualize LangGraph

the workflow architecture to show how each node connects in the final system

In [ ]:
# -----------------------------------
# Compile graph
# -----------------------------------
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

## Demo Queries
The LangGraph workflow was evaluated using multiple realistic laboratory coverage queries, demonstrating the system’s ability to parse clinical inputs, retrieve NCD policy evidence, apply deterministic coverage rules, and generate structured coverage explanations.

In [ ]:
# -----------------------------------
# NeedLe LangGraph Demonstration
# -----------------------------------

test_questions = [

    "Is CPT 83540 covered for ICD-10 R64 on 2026-01-01?",
    "Are serum iron studies covered for ICD-10 code R64 on 2026-01-01?",
    "What are the indications and limitations for serum iron studies?",
    "What can NeedLe help me do?",
    "Is thyroid stimulating hormone covered for ICD-10 E03.9 on 2026-01-01?",

]

print("\nNeedLe LangGraph Demonstration")
print("="*90)

for i, q in enumerate(test_questions, 1):

    state = {
        "question": q,
        "intent": "",
        "cpt_code": "",
        "icd_code": "",
        "date_of_service": "",
        "test_name": "",
        "evidence": [],
        "coverage_result": {},
        "confidence": {},
        "final_answer": ""
    }

    result = graph.invoke(
        state,
        config={"configurable": {"thread_id": f"needle-demo-{i}"}}
    )

    print("\n" + "-"*90)
    print(f"Query {i}: {q}")
    print("-"*90)
    print(result.get("final_answer"))


print("NeedLe demo completed successfully.")

## Demo Summary

The LangGraph demonstration shows that NeedLE AI can:

*   Perform deterministic CPT–ICD coverage validation
*   Identify the correct NCD policy for test-name queries
*   Retrieve supporting policy sections such as descriptions, indications, and limitations
*   Answer policy-only questions without requiring CPT codes
*   Respond to general user questions about system capabilities
*   Generate structured responses with confidence scores for transparency

These examples demonstrate the hybrid architecture combining rule-based validation with retrieval-augmented policy reasoning.

## Evaluation

To test multiple scenarios including covered cases, non-covered cases, and policy retrieval queries

In [ ]:
# ============================================================
# Batch Evaluation of NeedLe LangGraph
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("langgraph").setLevel(logging.ERROR)

eval_cases = [
    {
        "question": "Is CPT 87086 covered for ICD-10 A02.1 on 2026-01-01?",
        "expected": "covered"
    },
    {
        "question": "Is urine culture covered for ICD-10 Z00.00 on 2026-01-01?",
        "expected": "not covered"
    },
    {
        "question": "What are the indications and limitations for urine culture?",
        "expected": "policy retrieval"
    },
    {
        "question": "Is thyroid stimulating hormone covered for ICD-10 E03.9 on 2026-01-01?",
        "expected": "covered"
    },
]

rows = []

for i, case in enumerate(eval_cases, start=1):

    state = {
        "question": case["question"],
        "intent": "",
        "cpt_code": "",
        "icd_code": "",
        "date_of_service": "",
        "test_name": "",
        "evidence": [],
        "coverage_result": {},
        "confidence": {},
        "final_answer": ""
    }

    result = graph.invoke(
        state,
        config={"configurable": {"thread_id": f"needle-eval-{i}"}}
    )

    cov = result.get("coverage_result", {})
    conf = result.get("confidence", {})

    # Handle nested structures
    nested_cov = cov.get("coverage", {})
    results_list = cov.get("results", [])

    first_result = results_list[0] if results_list else {}

    rows.append({
        "question": case["question"],
        "expected": case["expected"],
        "intent_detected": result.get("intent", ""),

        "identified_ncd": (
            cov.get("ncd_number")
            or nested_cov.get("ncd_number")
            or first_result.get("ncd_number")
            or ""
        ),

        "coverage_found": (
            cov.get("found")
            if "found" in cov
            else nested_cov.get("found", "")
        ),

        "resolution_code": (
            cov.get("resolution_code")
            if "resolution_code" in cov
            else nested_cov.get("resolution_code")
            or first_result.get("resolution_code", "")
        ),

        "confidence_score": conf.get("score", "")
    })




eval_df = pd.DataFrame(rows)

eval_df

## Evaluation Summary

*   The evaluation results demonstrate that NeedLE successfully handled multiple query types across representative test scenarios. The system correctly identified coverage-related and policy-retrieval intents, mapped queries to the appropriate NCD policies, and returned structured coverage decisions when sufficient CPT–ICD rules were available.

*   It correctly identified both covered scenarios (CPT 87086 with ICD-10 A02.1 and thyroid testing with ICD-10 E03.9) and a non-covered scenario (urine culture with ICD-10 Z00.00). The system also successfully retrieved policy sections for informational queries involving indications and limitations.

*   Confidence scores were generated for all responses to improve transparency and support auditability. These results demonstrate that NeedLE can combine deterministic rule validation with policy retrieval to support explainable pre-claim laboratory coverage decision workflows.


## Save Outputs
- LangGraph architecture diagram
- evaluation results
- demo transcripts
- sample outputs

In [ ]:
import os
import json

# --------------------------------------------------
# Create output folder in Google Drive
# --------------------------------------------------
save_path = "/content/drive/MyDrive/NeedLe_outputs"
os.makedirs(save_path, exist_ok=True)

print("Saving outputs to:", save_path)

# --------------------------------------------------
# Save core artifacts
# --------------------------------------------------
with open(os.path.join(save_path, "ncd_index.json"), "w", encoding="utf-8") as f:
    json.dump(ncd_index, f, indent=2, ensure_ascii=False)

with open(os.path.join(save_path, "ncd_verbiage_sections.json"), "w", encoding="utf-8") as f:
    json.dump(filtered_records, f, indent=2, ensure_ascii=False)

with open(os.path.join(save_path, "cpt_to_ncd_map.json"), "w", encoding="utf-8") as f:
    json.dump(cpt_to_ncd, f, indent=2, ensure_ascii=False)

with open(os.path.join(save_path, "cpt_description_lookup.json"), "w", encoding="utf-8") as f:
    json.dump(cpt_desc_lookup, f, indent=2, ensure_ascii=False)

print("Saved knowledge artifacts")

# --------------------------------------------------
# Save evaluation results
# --------------------------------------------------
eval_df.to_csv(
    os.path.join(save_path, "needle_langgraph_evaluation_results.csv"),
    index=False
)

print("Saved evaluation results")

# --------------------------------------------------
# Save LangGraph architecture diagram
# --------------------------------------------------
img = graph.get_graph().draw_mermaid_png()

with open(os.path.join(save_path, "needle_langgraph_architecture.png"), "wb") as f:
    f.write(img)

print("Saved LangGraph architecture")

# --------------------------------------------------
# Save one sample response
# --------------------------------------------------
sample_question = "Is CPT 83540 covered for ICD-10 R64 on 2026-01-01?"

sample_state = {
    "question": sample_question,
    "intent": "",
    "cpt_code": "",
    "icd_code": "",
    "date_of_service": "",
    "test_name": "",
    "evidence": [],
    "coverage_result": {},
    "confidence": {},
    "final_answer": ""
}

sample_result = graph.invoke(
    sample_state,
    config={"configurable": {"thread_id": "needle-sample-output"}}
)

sample_path = os.path.join(save_path, "needle_sample_output.txt")

with open(sample_path, "w", encoding="utf-8") as f:
    f.write(sample_result["final_answer"])

print("Saved sample output")

# --------------------------------------------------
# Save full demo transcript
# --------------------------------------------------
demo_path = os.path.join(save_path, "needle_demo_output.txt")

with open(demo_path, "w", encoding="utf-8") as f:
    for i, q in enumerate(test_questions, 1):

        state = {
            "question": q,
            "intent": "",
            "cpt_code": "",
            "icd_code": "",
            "date_of_service": "",
            "test_name": "",
            "evidence": [],
            "coverage_result": {},
            "confidence": {},
            "final_answer": ""
        }

        result = graph.invoke(
            state,
            config={"configurable": {"thread_id": f"needle-save-{i}"}}
        )

        f.write(f"Query {i}: {q}\n\n")
        f.write(result["final_answer"])
        f.write("\n\n" + "=" * 80 + "\n\n")

print("Saved full demo transcript")
print("\nAll NeedLe outputs saved successfully.")

Overall, this project demonstrates how retrieval-augmented generation and deterministic healthcare policy rules can be combined to support explainable laboratory coverage decision workflows.”

**Files Generated and Submitted**

**Input Datasets**
*   CMS NCD Policy Manual PDF: Source document used for policy retrieval and section extraction
*   CPT–ICD Coverage Spreadsheet: Structured CMS coverage rules dataset used for deterministic coverage validation
*   ICD-10-CM Description Dataset: Used for ICD code validation and diagnosis descriptions
*   CPT Description Dataset: Used for CPT descriptions and CPT-to-NCD mapping

**Output Files**
*   Evaluation Results (.csv)
*   Sample Output (.txt)
*   Demo Transcript (.txt)




**The End**